In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F

from PIL import Image
from typing import Tuple, Dict, List

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

DATA_ROOT = "data"
os.makedirs(DATA_ROOT, exist_ok=True)

DATASET_DIR = os.path.join(DATA_ROOT, "PennFudanPed")

if not os.path.isdir(DATASET_DIR):
    !wget -q https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip -O {DATA_ROOT}/PennFudanPed.zip
    !unzip -q {DATA_ROOT}/PennFudanPed.zip -d {DATA_ROOT}
else:
    print("PennFudanPed уже скачан")


In [ ]:
class PennFudanDataset(Dataset):
    def __init__(self, root: str, transforms=None):
        self.root = root
        self.transforms = transforms
        
        # Пути к файлам изображений
        self.img_dir = os.path.join(root, "PennFudanPed", "PNGImages")
        self.mask_dir = os.path.join(root, "PennFudanPed", "PedMasks")
        
        self.imgs = list(sorted(os.listdir(self.img_dir)))
        self.masks = list(sorted(os.listdir(self.mask_dir)))
        
        self.imgs = [f for f in self.imgs if f.endswith(".png")]
        self.masks = [f for f in self.masks if f.endswith(".png")]
        
        
    def __len__(self) -> int:
        return len(self.imgs)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        
        mask = np.array(mask)
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[1:]  # убираем фон (0)
        
        masks = mask == obj_ids[:, None, None]
        
        num_objs = len(obj_ids)
        boxes = []
        for i in range(num_objs):
            pos = np.where(masks[i])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            boxes.append([xmin, ymin, xmax, ymax])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((num_objs,), dtype=torch.int64)  # 1 — класс "пешеход"
        masks = torch.as_tensor(masks, dtype=torch.uint8)
        
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
        
        target = {
            "boxes": boxes,       # [N, 4]
            "labels": labels,     # [N]
            "masks": masks,       # [N, H, W]
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        
        if self.transforms is not None:
            img, target = self.transforms(img, target)
        
        return img, target


In [ ]:
class ComposeTransforms:
    def __init__(self, transforms):
        self.transforms = transforms
        
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target

class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p
        
    def __call__(self, image, target):
        if np.random.rand() < self.p:
            image = F.hflip(image)
            w = image.shape[-1]
            boxes = target["boxes"]

            boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
            target["boxes"] = boxes
            if "masks" in target:
                target["masks"] = target["masks"].flip(-1)
        return image, target


In [ ]:
full_dataset = PennFudanDataset(DATA_ROOT, transforms=ComposeTransforms([
    ToTensor(),
    RandomHorizontalFlip(0.5)
]))


val_dataset = PennFudanDataset(DATA_ROOT, transforms=ComposeTransforms([
    ToTensor()
]))

len(full_dataset), len(val_dataset)


In [ ]:
indices = torch.randperm(len(full_dataset)).tolist()
split = int(0.8 * len(indices))
train_indices = indices[:split]
val_indices = indices[split:]

train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset   = torch.utils.data.Subset(val_dataset,   val_indices)

len(train_dataset), len(val_dataset)


In [ ]:
def plot_sample(dataset, idx=0):
    img, target = dataset[idx]
    img_np = img.permute(1, 2, 0).numpy()
    
    boxes = target["boxes"].numpy()
    
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img_np)
    for (xmin, ymin, xmax, ymax) in boxes:
        rect = plt.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False, color="red", linewidth=2
        )
        ax.add_patch(rect)
    ax.set_title(f"Image {idx}, {boxes.shape[0]} pedestrians")
    ax.axis("off")
    plt.show()

plot_sample(train_dataset, idx=0)


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_detection_model(num_classes: int):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

num_classes = 2  # фон + пешеход
model = get_detection_model(num_classes)
model.to(device)


In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10):
    model.train()
    loss_hist = []
    
    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        loss_hist.append(losses.item())
        
        if (i + 1) % print_freq == 0:
            print(f"Epoch {epoch}, Iter {i+1}/{len(data_loader)}, loss = {losses.item():.4f}")
    
    return np.mean(loss_hist)


In [ ]:
@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    model.eval()
    loss_hist = []
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        loss_hist.append(losses.item())
        
    return np.mean(loss_hist)


In [ ]:
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss = evaluate_loss(model, val_loader, device)
    lr_scheduler.step()
    
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")


In [ ]:
@torch.no_grad()
def predict_on_images(model, data_loader, device, score_thresh=0.5, max_images=5):
    model.eval()
    shown = 0
    
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        
        for img, target, output in zip(images, targets, outputs):
            if shown >= max_images:
                return
            
            img_np = img.cpu().permute(1, 2, 0).numpy()
            gt_boxes = target["boxes"].cpu().numpy()
            
            pred_boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            
            keep = scores >= score_thresh
            pred_boxes = pred_boxes[keep]
            scores = scores[keep]
            
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            ax.imshow(img_np)
            
            for (xmin, ymin, xmax, ymax) in gt_boxes:
                rect = plt.Rectangle(
                    (xmin, ymin),
                    xmax - xmin, ymax - ymin,
                    fill=False, color="lime", linewidth=2
                )
                ax.add_patch(rect)
            
            for (xmin, ymin, xmax, ymax), s in zip(pred_boxes, scores):
                rect = plt.Rectangle(
                    (xmin, ymin),
                    xmax - xmin, ymax - ymin,
                    fill=False, color="red", linewidth=2
                )
                ax.add_patch(rect)
                ax.text(
                    xmin, ymin,
                    f"{s:.2f}",
                    color="red",
                    fontsize=8,
                    bbox=dict(facecolor="white", alpha=0.5)
                )
            
            ax.set_title(f"GT (green) vs Pred (red), {len(pred_boxes)} detections")
            ax.axis("off")
            plt.show()
            
            shown += 1

predict_on_images(model, val_loader, device, score_thresh=0.5, max_images=5)


In [ ]:
def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    # boxes: [N, 4], формат (xmin, ymin, xmax, ymax)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])   # [N, M, 2]
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])   # [N, M, 2]
    
    wh = (rb - lt).clamp(min=0)  # [N, M, 2]
    inter = wh[:, :, 0] * wh[:, :, 1]
    
    union = area1[:, None] + area2 - inter
    iou = inter / union
    return iou

@torch.no_grad()
def simple_detection_metrics(model, data_loader, device, iou_thresh=0.5, score_thresh=0.5):
    model.eval()
    
    TP = 0
    FP = 0
    FN = 0
    
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        
        for target, output in zip(targets, outputs):
            gt_boxes = target["boxes"].to(device)
            pred_boxes = output["boxes"].to(device)
            scores = output["scores"].to(device)
            
            keep = scores >= score_thresh
            pred_boxes = pred_boxes[keep]
            
            if len(gt_boxes) == 0 and len(pred_boxes) == 0:
                continue
            if len(gt_boxes) == 0:
                FP += len(pred_boxes)
                continue
            if len(pred_boxes) == 0:
                FN += len(gt_boxes)
                continue
            
            ious = box_iou(gt_boxes, pred_boxes)  # [N_gt, N_pred]
            
            # Жадное сопоставление GT ↔ предсказаний
            gt_matched = torch.zeros(len(gt_boxes), dtype=torch.bool, device=device)
            pred_matched = torch.zeros(len(pred_boxes), dtype=torch.bool, device=device)
            
            # Сортируем все пары по IoU по убыванию
            gt_idx, pred_idx = torch.nonzero(ious >= iou_thresh, as_tuple=True)
            pair_ious = ious[gt_idx, pred_idx]
            order = torch.argsort(pair_ious, descending=True)
            
            for k in order:
                g = gt_idx[k]
                p = pred_idx[k]
                if not gt_matched[g] and not pred_matched[p]:
                    gt_matched[g] = True
                    pred_matched[p] = True
                    TP += 1
            
            FP += (~pred_matched).sum().item()
            FN += (~gt_matched).sum().item()
    
    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    return {"TP": TP, "FP": FP, "FN": FN, "precision": precision, "recall": recall}

metrics = simple_detection_metrics(model, val_loader, device)
metrics
